In [0]:
# Conectar ao catalogo e schema Silver
spark.sql("USE CATALOG MVP_gastos_publicos")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, when, lit, to_date, concat_ws, lpad
from pyspark.sql.types import DecimalType, IntegerType

# Ler a tabela Bronze
df = spark.table("MVP_gastos_publicos.bronze.gastos_publicos")

#Fazendo as devidas limpezas e conversoes nos dados 
# Como o os valores financeiros do arquivo estão como STRING temos que converter para decimal para que depois consiga fazer os devidos cálculos

df_silver = (
    df
    .withColumn("SALDO_PLOA",
                when(trim(col("SALDO_PLOA")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("SALDO_PLOA")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("DOTACAO_INICIAL",
                when(trim(col("DOTACAO_INICIAL")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("DOTACAO_INICIAL")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("DOTACAO_ATUALIZADA",
                when(trim(col("DOTACAO_ATUALIZADA")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("DOTACAO_ATUALIZADA")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("DESPESAS_EMPENHADAS",
                when(trim(col("DESPESAS_EMPENHADAS")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("DESPESAS_EMPENHADAS")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("DESPESAS_LIQUIDADAS",
                when(trim(col("DESPESAS_LIQUIDADAS")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("DESPESAS_LIQUIDADAS")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("DESPESAS_PAGAS",
                when(trim(col("DESPESAS_PAGAS")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("DESPESAS_PAGAS")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("RESTOS_A_PAGAR_PAGOS",
                when(trim(col("RESTOS_A_PAGAR_PAGOS")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("RESTOS_A_PAGAR_PAGOS")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    .withColumn("PAGAMENTOS_TOTAIS",
                when(trim(col("PAGAMENTOS_TOTAIS")).isin(["", "-"]), lit(None))
                .otherwise(regexp_replace(regexp_replace(trim(col("PAGAMENTOS_TOTAIS")), "\\.", ""), ",", ".").cast(DecimalType(18, 2))))
    
    # Convertendo colunas de codigo para INTEGER
    .withColumn("ID_ANO", col("ID_ANO").cast(IntegerType()))
    .withColumn("ID_MES", col("ID_MES").cast(IntegerType()))

    # Padronizando o formato das datas
    .withColumn("data_referencia",
                to_date(concat_ws("-", col("ID_ANO"), lpad(col("ID_MES"), 2, "0"), lit("01")), "yyyy-MM-dd"))
)


display(df_silver.limit(5))

ID_ANO,ID_MES,MES_LANCAMENTO,ORGAO_CODIGO,ORGAO_DESCRICAO,UNIDADE_ORCAMENTARIA_CODIGO,UNIDADE_ORCAMENTARIA_DESCRICAO,CO_FONTE_RECURSO,NO_FONTE_RECURSO,ID_GRUPO_DESPESA_NADE,NO_GRUPO_DESPESA_NADE,CO_MOAP_NADE,NO_MOAP_NADE,ID_IN_RESULTADO_EOF,NO_IN_RESULTADO_EOF,ID_IN_TP_CREDITO_CEOR,NO_IN_TP_CREDITO_CEOR,ID_FUNCAO_PT,NO_FUNCAO_PT,ID_SUBFUNCAO_PT,NO_SUBFUNCAO_PT,ID_PROGRAMA_PT,NO_PROGRAMA_PT,ACAO,NO_ACAO,SALDO_PLOA,DOTACAO_INICIAL,DOTACAO_ATUALIZADA,DESPESAS_EMPENHADAS,DESPESAS_LIQUIDADAS,DESPESAS_PAGAS,RESTOS_A_PAGAR_PAGOS,PAGAMENTOS_TOTAIS,Poder_Orgao,Primaria_Financeira,Categoria_LC200,Categoria_RTN,data_referencia
2025,1,JAN/2025,1000,CAMARA DOS DEPUTADOS,1101,CAMARA DOS DEPUTADOS,000,RECURSOS LIVRES DA UNIAO,1,PESSOAL E ENCARGOS SOCIAIS,90,APLICACOES DIRETAS,0,FINANCEIRO,A,INICIAL (LOA),99,RESERVA DE CONTINGENCIA,999,RESERVA DE CONTINGENCIA,0999,RESERVA DE CONTINGENCIA,0Z00,RESERVA DE CONTINGENCIA - FINANCEIRA,1499201.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,CD,Despesa Financeira,Exce��o: Despesa Financeira,N/A,2025-01-01
2025,1,JAN/2025,1000,CAMARA DOS DEPUTADOS,1101,CAMARA DOS DEPUTADOS,000,RECURSOS LIVRES DA UNIAO,1,PESSOAL E ENCARGOS SOCIAIS,90,APLICACOES DIRETAS,1,PRIMARIO OBRIGATORIO,A,INICIAL (LOA),1,LEGISLATIVA,122,ADMINISTRACAO GERAL,0034,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,20TP,ATIVOS CIVIS DA UNIAO,3908000000.00,3582350000.00,3582350000.00,3582350000.00,268873373.47,195130667.31,0.00,195130667.31,CD,Prim�ria,Despesa sujeita: Pessoal e Encargos Sociais,2.2.1 - Ativo Civil,2025-01-01
2025,1,JAN/2025,1000,CAMARA DOS DEPUTADOS,1101,CAMARA DOS DEPUTADOS,000,RECURSOS LIVRES DA UNIAO,1,PESSOAL E ENCARGOS SOCIAIS,90,APLICACOES DIRETAS,1,PRIMARIO OBRIGATORIO,A,INICIAL (LOA),9,PREVIDENCIA SOCIAL,272,PREVIDENCIA DO REGIME ESTATUTARIO,0034,PROGRAMA DE GESTAO E MANUTENCAO DO PODER LEGISLATIVO,0181,APOSENTADORIAS E PENSOES CIVIS DA UNIAO,1834756361.00,1834756361.00,1834756361.00,1834756361.00,146554866.81,146554866.81,0.00,146554866.81,CD,Prim�ria,Despesa sujeita: Pessoal e Encargos Sociais,2.2.3 - Aposentadorias e pens�es civis,2025-01-01
2025,1,JAN/2025,1000,CAMARA DOS DEPUTADOS,1101,CAMARA DOS DEPUTADOS,000,RECURSOS LIVRES DA UNIAO,1,PESSOAL E ENCARGOS SOCIAIS,90,APLICACOES DIRETAS,1,PRIMARIO OBRIGATORIO,A,INICIAL (LOA),28,ENCARGOS ESPECIAIS,846,OUTROS ENCARGOS ESPECIAIS,0909,OPERACOES ESPECIAIS: OUTROS ENCARGOS ESPECIAIS,00S6,"BENEFICIO ESPECIAL - LEI N. 12.618, DE 2012",49980948.00,49980948.00,49980948.00,49980948.00,2716838.82,2716838.82,0.00,2716838.82,CD,Prim�ria,Despesa sujeita: Pessoal e Encargos Sociais,2.2.3 - Aposentadorias e pens�es civis,2025-01-01
2025,1,JAN/2025,1000,CAMARA DOS DEPUTADOS,1101,CAMARA DOS DEPUTADOS,000,RECURSOS LIVRES DA UNIAO,1,PESSOAL E ENCARGOS SOCIAIS,90,APLICACOES DIRETAS,1,PRIMARIO OBRIGATORIO,A,INICIAL (LOA),28,ENCARGOS ESPECIAIS,846,OUTROS ENCARGOS ESPECIAIS,0909,OPERACOES ESPECIAIS: OUTROS ENCARGOS ESPECIAIS,00UX,DEMAIS APOSENTADORIAS E COMPLEMENTACOES,125000000.00,125000000.00,125000000.00,125000000.00,9053479.15,9053479.15,0.00,9053479.15,CD,Prim�ria,Despesa sujeita: Pessoal e Encargos Sociais,2.2.3 - Aposentadorias e pens�es civis,2025-01-01


In [0]:
# Salvar a tabela no schema silver 
df_silver.write.format("delta").mode("overwrite").saveAsTable("MVP_gastos_publicos.silver.gastos_publicos")

In [0]:
%sql
-- Visualizando os dados limpos
SELECT 
    ID_ANO,
    ID_MES,
    data_referencia,
    ORGAO_DESCRICAO,
    PAGAMENTOS_TOTAIS,
    DESPESAS_EMPENHADAS
FROM MVP_gastos_publicos.silver.gastos_publicos
LIMIT 10;

ID_ANO,ID_MES,data_referencia,ORGAO_DESCRICAO,PAGAMENTOS_TOTAIS,DESPESAS_EMPENHADAS
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,225.00,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,15828976.60
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,1024768.40,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,89.34,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,null
2024,1,2024-01-01,CAMARA DOS DEPUTADOS,null,null
